# ***Libraries Import***

In [6]:
!pip install noisereduce

In [7]:
import os
import librosa
import librosa.display
import numpy as np
import seaborn as sns
import matplotlib.pyplot as plt
import noisereduce as nr
from sklearn.metrics import accuracy_score
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from PIL import Image
import IPython.display as ipd

# ***Load Audio***

In [8]:
audio_eg = "/kaggle/input/audio-cats-and-dogs/cats_dogs/train/cat/cat_119.wav"
ipd.Audio(audio_eg)

ValueError: rate must be specified when data is a numpy array or list of audio samples.

In [ ]:
# Load an audio file
audio_path = "/kaggle/input/audio-cats-and-dogs/cats_dogs/train/cat/cat_119.wav"
y, sr = librosa.load(audio_path, sr=None)
y

# ***Waveform Plotting***

In [ ]:
plt.figure(figsize=(12, 6))
# Waveform Plotting
plt.subplot(2, 1, 1)
librosa.display.waveshow(y, sr=sr, alpha=0.7)
plt.title("Waveform of the Audio Signal")
plt.xlabel("Time (seconds)")
plt.ylabel("Amplitude")

plt.show()

In [ ]:
import seaborn as sns
import matplotlib.pyplot as plt
import pandas as pd
# Combine labels
label_names = {0: 'Cat', 1: 'Dog'}
train_labels_named = [label_names[i] for i in y_train]
test_labels_named = [label_names[i] for i in y_test]
# Plot class distribution
sns.countplot(x=train_labels_named)
plt.title("Training Set: Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()

sns.countplot(x=test_labels_named)
plt.title("Test Set: Class Distribution")
plt.xlabel("Class")
plt.ylabel("Count")
plt.show()

# ***Geometrical Feature Extraction***

In [ ]:
def extract_features_from_directory(directory_path, label):
    features = []
    labels = []

    for file in os.listdir(directory_path):
        if file.endswith('.wav'):
            try:
                file_path = os.path.join(directory_path, file)
                y_audio, sr = librosa.load(file_path, sr=None)
                y_audio = nr.reduce_noise(y=y_audio, sr=sr)
                y_trimmed, _ = librosa.effects.trim(y_audio)
                y_normalized = y_trimmed / max(abs(y_trimmed))
                mfccs = librosa.feature.mfcc(y=y_normalized, sr=sr, n_mfcc=13)

                mfccs_mean = np.mean(mfccs, axis=1)
                mfccs_std = np.std(mfccs, axis=1)
                mfccs_min = np.min(mfccs, axis=1)
                mfccs_max = np.max(mfccs, axis=1)
                mfcc_vector = np.hstack([mfccs_mean, mfccs_std, mfccs_min, mfccs_max])

                features.append(mfcc_vector)
                labels.append(label)
            except Exception as e:
                print(f"Error processing {file}: {e}")

    return features, labels

X = np.array(X)
y = np.array(y)

# Create DataFrame with feature names
feature_names = [f"mfcc{i+1}_{stat}" for stat in ['mean', 'std', 'min', 'max'] for i in range(13)]
df = pd.DataFrame(X, columns=feature_names)

# correlation matrix
corr_matrix = df.corr().abs()

# visualize
plt.figure(figsize=(14, 10))
sns.heatmap(corr_matrix, cmap='coolwarm', square=True, linewidths=0.5)
plt.title("Correlation Matrix of MFCC Features")
plt.show()

# Drop highly correlated features (threshold > 0.9)
upper = corr_matrix.where(np.triu(np.ones(corr_matrix.shape), k=1).astype(bool))
to_drop = [col for col in upper.columns if any(upper[col] > 0.9)]

# Reduce feature set
df_reduced = df.drop(columns=to_drop)
X_reduced = df_reduced.values

print(f"Original feature count: {df.shape[1]}")
print(f"Removed due to high correlation: {len(to_drop)}")
print(f"Remaining features: {X_reduced.shape[1]}")
print("Selected features:", df_reduced.columns.tolist())

In [ ]:
# Train data
train_cat_path = "/kaggle/input/audio-cats-and-dogs/cats_dogs/train/cat"
train_dog_path = "/kaggle/input/audio-cats-and-dogs/cats_dogs/train/dog"
X_cat_train, y_cat_train = extract_features_from_directory(train_cat_path, label=0)
X_dog_train, y_dog_train = extract_features_from_directory(train_dog_path, label=1)

X_train = np.array(X_cat_train + X_dog_train)
y_train = np.array(y_cat_train + y_dog_train)

# Test data
test_cat_path = "/kaggle/input/audio-cats-and-dogs/cats_dogs/test/cats"
test_dog_path = "/kaggle/input/audio-cats-and-dogs/cats_dogs/test/test"
X_cat_test, y_cat_test = extract_features_from_directory(test_cat_path, label=0)
X_dog_test, y_dog_test = extract_features_from_directory(test_dog_path, label=1)
X_test = np.array(X_cat_test + X_dog_test)
y_test = np.array(y_cat_test + y_dog_test)

In [ ]:
scaler = StandardScaler()
X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

model = RandomForestClassifier(n_estimators=100, random_state=42)
model.fit(X_train_scaled, y_train)
y_pred = model.predict(X_test_scaled)

In [9]:
print("Accuracy:", accuracy_score(y_test, y_pred)*100,'%')
print("\n Classification Report:\n", classification_report(y_test, y_pred, target_names=["Cat", "Dog"]))
print("\n Confusion Matrix:\n", confusion_matrix(y_test, y_pred))

NameError: name 'y_test' is not defined